<a href="https://colab.research.google.com/github/KatarinaNunes/Atividade_Disciplina/blob/main/Analise_docking_mleprae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!ls /content/drive/MyDrive


 0806-rm-197.en.pt.pdf
 0b36e19f3c2371d646a473c5d42cdcaf.pdf
 1261.full.en.pt.pdf
 15._Influ_ncia_do_sequenciamento_de_nova_gera_o_no_futuro_da_gen_tica_da_conserva_o.pdf
 1Rgk7DDd3e9w1XNNkE9J_019.613.722-58.pdf
 1-s2.0-S0042682207003236-main.en.pt.pdf
 1-s2.0-S221466361500005X-main.pdf
 20211013_174158.jpg
 20240328_153856.jpg
 20240904_093845.jpg
 20240904_093857.jpg
'2024-11-02 22-22-23.mkv'
'2024-11-02 22-34-21.mkv'
'2025-01-10 23-15-21.mkv'
'2025-08-02 20-57-46.mkv'
'2025-08-02 21-23-50.mkv'
 47847793.gdoc
 8cdf3c4f63ceb7bca070c065cf2a35fa.pdf
 9964e3ee28118469406c081b0e7ce968_jqhd3o8jmk44rmlflooegifo40_c_70_receitas_de_sabonetes.pdf
 a64b6d2c29017322948c7e1b240d8481_f9qet6da6s6hh6egoegaaaq9i5_c_lista_de_fornecedores.pdf
'ABSTRACT - XMEETING_rev.docx'
 ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
'Amostras Indígena Pará.gdoc'
 anotacao_prokka
 Apostila-cosmética-natural.pdf
 artigo
'artigos principais tcc'
 backup
'bes-nfev-qov (2021-04-23 at 11:37

In [7]:
!ls /content/drive/MyDrive/variants_mlep


pandadock_report_AC1082S.gsheet  pandadock_report_S456M.gsheet
pandadock_report_G448D.gsheet	 pandadock_report_T171N.gsheet
pandadock_report_H451Y.gsheet	 pandadock_report_T433I.gsheet
pandadock_report_N1111D.gsheet	 pandadock_report_WT.gsheet
pandadock_report_S456L.gsheet


In [8]:
!pip install --quiet gspread gspread_dataframe


In [17]:
from google.colab import auth
auth.authenticate_user()

import google.auth
creds, _ = google.auth.default()

from googleapiclient.discovery import build
drive_service = build("drive", "v3", credentials=creds)

!pip -q install gspread gspread_dataframe
import gspread
from gspread_dataframe import get_as_dataframe
gc = gspread.authorize(creds)


In [18]:
FOLDER_NAME = "variants_mlep"

q = (
    f"name='{FOLDER_NAME}' and "
    "mimeType='application/vnd.google-apps.folder' and "
    "'root' in parents and trashed=false"
)

res = drive_service.files().list(q=q, fields="files(id,name)").execute()
folders = res.get("files", [])

if not folders:
    raise RuntimeError("Não achei a pasta 'variants_mlep' no root do Meu Drive. Renomeou/moveu?")
folder_id = folders[0]["id"]
print("Folder ID:", folder_id)


Folder ID: 1cXs9ji1JsCQgSKSqS9uK4MIFsxQF0XTx


In [19]:
import pandas as pd

# Usa o folder_id que você já obteve:
folder_id = "1cXs9ji1JsCQgSKSqS9uK4MIFsxQF0XTx"

q = (
    f"'{folder_id}' in parents and "
    "mimeType='application/vnd.google-apps.spreadsheet' and "
    "name contains 'pandadock_report_' and trashed=false"
)

res = drive_service.files().list(q=q, fields="files(id,name)").execute()
sheets = res.get("files", [])

print("Planilhas encontradas:", len(sheets))
print(*[s["name"] for s in sheets], sep="\n")

all_dfs = []
errors = []

for s in sheets:
    name = s["name"]
    sid  = s["id"]
    variant = name.replace("pandadock_report_", "").strip()

    try:
        sh = gc.open_by_key(sid)
        ws = sh.sheet1  # primeira aba

        df = get_as_dataframe(ws, evaluate_formulas=True)
        df = df.dropna(how="all").dropna(axis=1, how="all")

        df["Variant"] = variant
        all_dfs.append(df)

    except Exception as e:
        errors.append((name, str(e)))

df_all = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

print("\nDFs lidos:", len(all_dfs))
print("Erros:", len(errors))
if errors:
    print("\nPrimeiros erros:")
    for it in errors[:5]:
        print(it)

df_all.head()


Planilhas encontradas: 9
pandadock_report_AC1082S
pandadock_report_G448D
pandadock_report_H451Y
pandadock_report_N1111D
pandadock_report_S456L
pandadock_report_S456M
pandadock_report_T171N
pandadock_report_T433I
pandadock_report_WT

DFs lidos: 9
Erros: 0


,Rank,Pose_ID,Score,Energy,Binding_Affinity,IC50_uM,EC50_uM,Ligand_Efficiency,VdW_Energy,Electrostatic_Energy,HBond_Energy,Hydrophobic_Energy,Solvation_Energy,Entropy_Energy,Clash_Score,HBond_Count,Hydrophobic_Count,Salt_Bridge_Count,Variant
0,1.0,pose_1,12.348674,-12.348674,-7.409204,3.70,37.0,-0.063327,0.074944,0.0,-6.0,-4.0,0.0,0.0,122.688277,0.0,0.0,0.0,AC1082S
1,2.0,pose_2,12.506461,-12.506461,-7.503877,3.16,31.6,-0.064136,0.075629,0.0,-6.0,-4.0,0.0,0.0,119.227255,0.0,0.0,0.0,AC1082S
2,3.0,pose_3,12.768492,-12.768492,-7.661095,2.42,24.2,-0.065479,0.075192,0.0,-6.0,-4.0,0.0,0.0,123.230790,0.0,0.0,0.0,AC1082S
3,4.0,pose_4,12.816959,-12.816959,-7.690175,2.30,23.0,-0.065728,0.074907,0.0,-6.0,-4.0,0.0,0.0,121.806830,0.0,0.0,0.0,AC1082S
4,5.0,pose_5,12.847558,-12.847558,-7.708535,2.23,22.3,-0.065885,0.075779,0.0,-6.0,-4.0,0.0,0.0,124.020830,0.0,0.0,0.0,AC1082S


In [20]:
df_all.to_csv("/content/drive/MyDrive/variants_mlep/pandadock_all_variants.csv", index=False)
print("OK: salvo no Drive em variants_mlep/pandadock_all_variants.csv")


OK: salvo no Drive em variants_mlep/pandadock_all_variants.csv


In [21]:
import pandas as pd
import re

in_path  = "/content/drive/MyDrive/variants_mlep/pandadock_all_variants.csv"
df = pd.read_csv(in_path)

# --- 1) Normaliza o nome da variante (Variant) ---
# Ex.: "S456M" -> "Ser456Met" (opcional) ou mantém "S456M" padronizado
aa1_to_aa3 = {
    "A":"Ala","R":"Arg","N":"Asn","D":"Asp","C":"Cys","Q":"Gln","E":"Glu","G":"Gly",
    "H":"His","I":"Ile","L":"Leu","K":"Lys","M":"Met","F":"Phe","P":"Pro","S":"Ser",
    "T":"Thr","W":"Trp","Y":"Tyr","V":"Val"
}

def clean_variant(v):
    if pd.isna(v):
        return None
    v = str(v).strip()

    # casos tipo WT, wt, WildType etc.
    if v.upper() in {"WT", "WILD", "WILDTYPE", "WILD_TYPE"}:
        return "WT"

    # remove espaços e caracteres estranhos
    v = re.sub(r"\s+", "", v)

    # padrão: S456M / T171N / N1111D etc
    m = re.match(r"^([A-Z])(\d+)([A-Z])$", v.upper())
    if m:
        ref, pos, alt = m.group(1), m.group(2), m.group(3)
        return f"{ref}{pos}{alt}"  # versão curta padronizada

    # se vier tipo "Ser456Met"
    m3 = re.match(r"^([A-Za-z]{3})(\d+)([A-Za-z]{3})$", v)
    if m3:
        ref3, pos, alt3 = m3.group(1).title(), m3.group(2), m3.group(3).title()
        # converte 3 letras -> 1 letra se possível
        aa3_to_aa1 = {v:k for k,v in aa1_to_aa3.items()}
        ref1 = aa3_to_aa1.get(ref3, "?")
        alt1 = aa3_to_aa1.get(alt3, "?")
        if ref1 != "?" and alt1 != "?":
            return f"{ref1}{pos}{alt1}"
        return f"{ref3}{pos}{alt3}"

    # fallback: devolve limpo
    return v

df["Variant_clean"] = df["Variant"].apply(clean_variant)

# (opcional) criar versão longa "Ser456Met" para figurinhas/relatório
def variant_to_long(v):
    if v == "WT" or pd.isna(v):
        return v
    m = re.match(r"^([A-Z])(\d+)([A-Z])$", str(v))
    if not m:
        return v
    ref, pos, alt = m.group(1), m.group(2), m.group(3)
    return f"{aa1_to_aa3.get(ref, ref)}{pos}{aa1_to_aa3.get(alt, alt)}"

df["Variant_long"] = df["Variant_clean"].apply(variant_to_long)

df[["Variant","Variant_clean","Variant_long"]].drop_duplicates().sort_values("Variant_clean").head(20)


,Variant,Variant_clean,Variant_long
0,AC1082S,AC1082S,AC1082S
50,G448D,G448D,Gly448Asp
100,H451Y,H451Y,His451Tyr
150,N1111D,N1111D,Asn1111Asp
200,S456L,S456L,Ser456Leu
250,S456M,S456M,Ser456Met
300,T171N,T171N,Thr171Asn
350,T433I,T433I,Thr433Ile
400,WT,WT,WT


In [ ]:
cols = list(df.columns)

# tenta achar colunas numéricas relevantes
numeric_cols = df.select_dtypes(include="number").columns.tolist()

print("Numéricas:", numeric_cols)

# candidatos típicos: score, affinity, energy, deltaG, ic50, kd...
candidates = [c for c in cols if re.search(r"(score|affin|energy|dg|delta|ic50|kd|kcal)", c, re.IGNORECASE)]
print("Candidatas:", candidates)
